# GRU · 14 Optional Extension and Demo

An optional, GRU-only extension for very short reviews, followed by a demo where you can try your own reviews. It is **not** part of the four-model comparison. **Saves:** `gru_short_model.keras`, `gru_short_extension_metrics.json`.

**Needs the trained GRU** saved by notebook 08 (`results/gru_model.keras`). If it is not found, it is trained automatically first (about 15 seconds, same settings and seed, so the same result).

*Notebook 14 of 14 · Previous: 13 Handover · Next: none (the end)*

## Setup (the same in every notebook)

Each notebook in this project is self-contained. The cells below define the group settings and only the helper functions this notebook uses. They are identical in every notebook and nothing is imported from another file, so the notebook also runs on its own, for example in Colab.

In [1]:
# ---- Imports, project folders and dataset location ----------------------------------------------
import sys, re, json, time, pathlib, textwrap
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tensorflow import keras
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)

layers = keras.layers
IN_COLAB = "google.colab" in sys.modules

# GRU_ROOT is the folder that holds the numbered step folders (01_Setup, 02_Data_Audit, ...).
# If this notebook is opened on its own (for example in Colab), the current folder is used instead.
here = pathlib.Path.cwd().resolve()
GRU_ROOT = next((p for p in [here, *here.parents] if (p / "01_Setup").is_dir()), here)
RESULTS = GRU_ROOT / "results"              # figures, metrics, predictions and trained models are saved here
RESULTS.mkdir(exist_ok=True)

# The dataset is the only input file.
DATA_NAME = "deceptive-opinion.csv"
search_in = [GRU_ROOT, GRU_ROOT / "dataset", GRU_ROOT.parent / "dataset", here, here / "dataset",
             pathlib.Path("/content")]
DATA_PATH = next((d / DATA_NAME for d in search_in if (d / DATA_NAME).exists()), None)

if DATA_PATH is None and IN_COLAB:
    from google.colab import files
    print(f"Please choose {DATA_NAME} from your computer...")
    uploaded = files.upload()                       # saved in the current folder
    DATA_PATH = here / next(iter(uploaded))
if DATA_PATH is None:
    raise FileNotFoundError(f"Cannot find {DATA_NAME}. Put it in a 'dataset' folder next to the step folders "
                            "(or next to this notebook) and run this cell again.")

print("Dataset:", DATA_PATH)
print("Results folder:", RESULTS)

Dataset: /Users/akalanka/Documents/GitHub/Deep_Learning_Assignment/Deep-Learning---Assignment-/dataset/deceptive-opinion.csv
Results folder: /Users/akalanka/Documents/GitHub/Deep_Learning_Assignment/Deep-Learning---Assignment-/gru/results


In [2]:
# ---- Group settings: identical for Simple RNN, LSTM, GRU and 1D CNN ------------------------------
SEED = 42

# Labels: this dataset has no star rating, so the `polarity` column is the label.
LABEL_COLUMN = "polarity"
LABEL_MAP = {"negative": 0, "positive": 1}

# Text -> integer sequences
VOCAB_SIZE = 5000          # total ids, including <PAD>=0 and <OOV>=1
MAX_LEN = 300              # tokens per review after cleaning
PAD_TOKEN, OOV_TOKEN = "<PAD>", "<OOV>"
PADDING = "pre"            # zeros in front, so a recurrent layer ends on real words
TRUNCATING = "post"        # long reviews keep their first MAX_LEN tokens

# Training
EMBED_DIM = 64
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_EPOCHS = 30
PATIENCE = 5               # EarlyStopping patience, monitoring validation loss
MONITOR = "val_loss"
THRESHOLD = 0.5            # probability cut-off used for every model's test metrics

# Fixed 80/10/10 split (stratified on the label, seed 42, made after removing duplicates).
# Numbers are 0-based row positions in deceptive-opinion.csv. Every row not listed here is a TRAINING row.
# All four models must use exactly these two lists.
VAL_IDS = {
    3, 4, 6, 7, 14, 24, 25, 27, 33, 38, 46, 50, 60, 66, 72, 74, 77, 85, 89, 92, 106, 110, 112, 136,
    144, 172, 211, 215, 224, 230, 238, 242, 248, 252, 288, 302, 311, 319, 336, 345, 368, 376, 385,
    393, 402, 404, 434, 440, 442, 465, 473, 510, 512, 515, 517, 524, 533, 536, 542, 549, 553, 558,
    579, 581, 594, 603, 624, 630, 638, 644, 645, 661, 663, 665, 699, 713, 729, 754, 760, 788, 801,
    813, 820, 821, 827, 832, 840, 864, 873, 893, 897, 900, 904, 907, 908, 923, 930, 936, 962, 963,
    989, 991, 1002, 1008, 1017, 1043, 1045, 1054, 1073, 1098, 1112, 1119, 1134, 1143, 1170, 1189,
    1191, 1201, 1239, 1258, 1262, 1263, 1264, 1270, 1280, 1295, 1308, 1309, 1314, 1328, 1344, 1350,
    1359, 1360, 1364, 1367, 1368, 1369, 1370, 1377, 1378, 1381, 1418, 1423, 1450, 1467, 1479, 1480,
    1490, 1505, 1506, 1522, 1547, 1551, 1552, 1573, 1578, 1588, 1593, 1594
}
TEST_IDS = {
    0, 2, 5, 9, 39, 45, 47, 55, 108, 117, 120, 125, 138, 139, 145, 171, 191, 193, 213, 222, 227,
    228, 232, 256, 258, 260, 272, 273, 276, 281, 299, 314, 321, 325, 349, 362, 364, 391, 409, 424,
    436, 445, 467, 469, 484, 502, 503, 505, 507, 523, 525, 531, 545, 547, 572, 573, 605, 606, 612,
    622, 623, 633, 669, 671, 681, 687, 691, 698, 718, 725, 731, 741, 752, 753, 765, 770, 771, 779,
    784, 785, 804, 814, 834, 835, 847, 852, 859, 866, 882, 887, 889, 901, 932, 940, 958, 961, 968,
    972, 993, 1003, 1007, 1019, 1020, 1033, 1044, 1046, 1055, 1072, 1076, 1079, 1091, 1117, 1141,
    1147, 1149, 1151, 1176, 1182, 1183, 1193, 1195, 1196, 1205, 1210, 1217, 1222, 1231, 1259, 1265,
    1275, 1278, 1279, 1288, 1293, 1296, 1302, 1306, 1312, 1324, 1357, 1365, 1366, 1396, 1404, 1416,
    1446, 1451, 1465, 1478, 1485, 1487, 1492, 1503, 1504, 1516, 1524, 1540, 1562, 1582, 1597
}
keras.utils.set_random_seed(SEED)          # seeds Python, NumPy and TensorFlow
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| seed", SEED)
print("Devices:", [d.device_type for d in tf.config.list_physical_devices()])

TensorFlow 2.21.0 | Keras 3.15.1 | seed 42
Devices: ['CPU']


In [3]:
# ---- Data pipeline: load -> label -> de-duplicate -> split -> clean -> vocabulary -> pad ----------
# Leakage rule: the vocabulary and the class weights are built from TRAINING rows only.
# Validation and test rows are only ever encoded with the training vocabulary.

def load_labelled_data() -> pd.DataFrame:
    """Read the CSV, map polarity -> 0/1 and drop exact duplicate reviews.

    Duplicates must go before splitting, otherwise a copy of a training review could land in the
    test set and the test set would no longer be unseen.
    """
    df = pd.read_csv(DATA_PATH)
    df["row_id"] = np.arange(len(df))                   # 0-based row number in the CSV
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df[LABEL_COLUMN].map(LABEL_MAP)
    assert df["label"].notna().all(), "unexpected label value"
    df["label"] = df["label"].astype(int)
    df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
    return df[["row_id", "text", "label", "hotel", "deceptive", "source"]]


def assign_split(df: pd.DataFrame) -> np.ndarray:
    """'train' / 'val' / 'test' for every row, from the fixed id lists above."""
    if not (VAL_IDS | TEST_IDS) <= set(df["row_id"]):
        raise ValueError("The fixed validation/test ids do not match this dataset. Was the CSV changed?")
    return np.where(df["row_id"].isin(TEST_IDS), "test",
                    np.where(df["row_id"].isin(VAL_IDS), "val", "train"))


# Text cleaning: identical for all models; a pure per-review function, nothing is fitted.
_HTML = re.compile(r"<[^>]+>")
_URL = re.compile(r"(https?://\S+|www\.\S+)")
_CONTRACTIONS = [                       # keep negation: "didn't" -> "did not", not "didn t"
    (r"\bwon't\b", "will not"), (r"\bcan't\b", "can not"), (r"\bcannot\b", "can not"),
    (r"n't\b", " not"), (r"'re\b", " are"), (r"'ve\b", " have"), (r"'ll\b", " will"),
    (r"'d\b", " would"), (r"'m\b", " am"), (r"'s\b", ""),
]
_NON_LETTER = re.compile(r"[^a-z\s]")
_SPACES = re.compile(r"\s+")


def clean_text(text: str) -> str:
    """lowercase -> strip HTML -> strip URLs -> expand contractions -> keep letters only.

    Stop-words are deliberately NOT removed: words such as "not", "no" and "never" flip the sentiment.
    """
    text = text.lower().replace("’", "'")
    text = _HTML.sub(" ", text)
    text = _URL.sub(" ", text)
    for pattern, replacement in _CONTRACTIONS:
        text = re.sub(pattern, replacement, text)
    text = _NON_LETTER.sub(" ", text)           # digits, punctuation, symbols
    return _SPACES.sub(" ", text).strip()


def build_vocab(train_token_lists, vocab_size: int = VOCAB_SIZE) -> dict:
    """Most frequent TRAINING words -> ids. 0 = <PAD>, 1 = <OOV>, real words start at 2."""
    counts = Counter(tok for tokens in train_token_lists for tok in tokens)
    words = [w for w, _ in counts.most_common(vocab_size - 2)]
    vocab = {PAD_TOKEN: 0, OOV_TOKEN: 1}
    vocab.update({w: i + 2 for i, w in enumerate(words)})
    return vocab


def encode(tokens, vocab: dict) -> list:
    oov = vocab[OOV_TOKEN]
    return [vocab.get(t, oov) for t in tokens]


def decode(ids, vocab: dict, skip_pad: bool = True) -> list:
    """Inverse of `encode`: ids -> words. Unknown words come back as <OOV> (the original is lost)."""
    inverse = {i: w for w, i in vocab.items()}
    return [inverse[int(i)] for i in ids if not (skip_pad and int(i) == 0)]


def pad(sequences, max_len: int = MAX_LEN) -> np.ndarray:
    out = np.zeros((len(sequences), max_len), dtype="int32")
    for i, seq in enumerate(sequences):
        seq = seq[:max_len] if TRUNCATING == "post" else seq[-max_len:]
        if PADDING == "pre":
            out[i, max_len - len(seq):] = seq
        else:
            out[i, :len(seq)] = seq
    return out


def compute_class_weights(y_train) -> dict:
    """weight_c = n_samples / (n_classes * n_samples_in_class_c)   (same as sklearn 'balanced')."""
    classes, counts = np.unique(y_train, return_counts=True)
    return {int(c): float(len(y_train) / (len(classes) * n)) for c, n in zip(classes, counts)}


@dataclass
class Data:
    X_train: np.ndarray
    y_train: np.ndarray
    X_val: np.ndarray
    y_val: np.ndarray
    X_test: np.ndarray
    y_test: np.ndarray
    vocab: dict
    class_weight: dict
    train_df: pd.DataFrame     # raw + cleaned text, kept for EDA / error analysis
    val_df: pd.DataFrame
    test_df: pd.DataFrame


def prepare_data() -> Data:
    """Everything a model needs, in one call."""
    df = load_labelled_data()
    df["split"] = assign_split(df)
    df["clean"] = df["text"].map(clean_text)
    df["tokens"] = df["clean"].str.split()

    parts = {name: g.reset_index(drop=True) for name, g in df.groupby("split")}
    vocab = build_vocab(parts["train"]["tokens"].tolist())          # <- TRAIN ONLY

    def to_xy(part: pd.DataFrame):
        seqs = [encode(tokens, vocab) for tokens in part["tokens"]]
        return pad(seqs), part["label"].to_numpy(dtype="int32")

    X_train, y_train = to_xy(parts["train"])
    X_val, y_val = to_xy(parts["val"])
    X_test, y_test = to_xy(parts["test"])
    return Data(X_train, y_train, X_val, y_val, X_test, y_test, vocab,
                compute_class_weights(y_train), parts["train"], parts["val"], parts["test"])

In [4]:
# ---- Evaluation helpers: the same metrics and result format for all four models ------------------
# Positive class (label 1) is the class used for precision / recall / F1.

def compute_metrics(y_true, y_prob, threshold: float = THRESHOLD) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).ravel()
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        # per-class view, needed for the error analysis
        "negative_precision": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        "negative_recall": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "negative_f1": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def bootstrap_ci(y_true, y_prob, n_boot: int = 2000, seed: int = SEED) -> dict:
    """95% bootstrap interval for accuracy / F1 / ROC-AUC on the test set.
    It only describes how uncertain the (small) test set makes the numbers. It is NOT used to tune anything."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).ravel()
    rng = np.random.default_rng(seed)
    out = {"accuracy": [], "f1": [], "roc_auc": []}
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        m = compute_metrics(y_true[idx], y_prob[idx])
        for k in out:
            out[k].append(m[k])
    return {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in out.items()}


def save_result(path, model_name: str, test_metrics: dict, *, training_time_sec: float,
                trainable_params: int, total_params: int, epochs_run: int, best_epoch: int,
                extra: dict = None) -> dict:
    """One JSON file per model with the same keys for every model, for the group's comparison table."""
    row = {
        "model": model_name,
        "accuracy": test_metrics["accuracy"], "precision": test_metrics["precision"],
        "recall": test_metrics["recall"], "f1": test_metrics["f1"], "roc_auc": test_metrics["roc_auc"],
        "training_time_sec": training_time_sec,
        "trainable_params": trainable_params, "total_params": total_params,
        "epochs_run": epochs_run, "best_epoch": best_epoch,
        "seed": SEED, "max_len": MAX_LEN, "vocab_size": VOCAB_SIZE,
        "embed_dim": EMBED_DIM, "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE, "threshold": THRESHOLD,
        "confusion_matrix": {k: test_metrics[k] for k in ("tn", "fp", "fn", "tp")},
    }
    if extra:
        row.update(extra)
    with open(path, "w") as f:
        json.dump(row, f, indent=2)
    return row

In [5]:
# ---- Model helpers (the same functions as in notebooks 07 and 08) ----
def build_gru():
    return keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(VOCAB_SIZE, EMBED_DIM),
        layers.GRU(64),
        layers.Dropout(0.5),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ], name="gru_sentiment")

def n_params(weights):
    return int(sum(np.prod(w.shape) for w in weights))

def train_gru(seed, verbose=2):
    keras.utils.set_random_seed(seed)
    m = build_gru()
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
              loss="binary_crossentropy", metrics=["accuracy"])
    early_stop = keras.callbacks.EarlyStopping(monitor=MONITOR, patience=PATIENCE,
                                               restore_best_weights=True)
    start = time.perf_counter()
    hist = m.fit(data.X_train, data.y_train,
                 validation_data=(data.X_val, data.y_val),          # validation only, never test
                 epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
                 class_weight=data.class_weight, callbacks=[early_stop], verbose=verbose)
    return m, hist.history, time.perf_counter() - start

def save_training_outputs(model, history, train_time, epochs_run, best_epoch):
    # what the later notebooks need to continue without retraining
    model.save(RESULTS / "gru_model.keras")
    (pd.DataFrame(history).rename_axis("epoch").reset_index().assign(epoch=lambda d: d.epoch + 1)
       .to_csv(RESULTS / "gru_history.csv", index=False))
    (RESULTS / "gru_training_info.json").write_text(json.dumps(
        {"train_time_sec": train_time, "epochs_run": epochs_run, "best_epoch": best_epoch}))
def load_or_train_main():
    # Returns (model, history, train_time, epochs_run, best_epoch) for the main GRU (seed 42).
    # It uses the files saved by notebook 08 when they exist. Otherwise it trains the GRU here
    # (about 15 seconds, same settings and seed, so the result is the same). Needs `data` to exist.
    paths = [RESULTS / n for n in ("gru_model.keras", "gru_history.csv", "gru_training_info.json")]
    if all(p.exists() for p in paths):
        info = json.loads(paths[2].read_text())
        history = pd.read_csv(paths[1]).drop(columns="epoch").to_dict("list")
        print("Loaded the trained GRU saved by notebook 08.")
        return keras.models.load_model(paths[0]), history, info["train_time_sec"], info["epochs_run"], info["best_epoch"]
    print("No saved GRU found in results/, so it is trained here (same settings as notebook 08)...")
    model, history, train_time = train_gru(SEED, verbose=0)
    epochs_run, best_epoch = len(history["loss"]), int(np.argmin(history["val_loss"])) + 1
    save_training_outputs(model, history, train_time, epochs_run, best_epoch)
    return model, history, train_time, epochs_run, best_epoch

### Load what this notebook needs

In [6]:
data = prepare_data()
model, history, train_time, epochs_run, best_epoch = load_or_train_main()

Loaded the trained GRU saved by notebook 08.


## Optional extension: short-review robustness (GRU only)

**This notebook is not part of the four-model comparison.** The main GRU (notebooks 07 to 11) uses the shared training set and stays the model reported in the group table.

**Problem.** Every training review has at least 25 words, and positive reviews are shorter on average, so the main GRU has learned that "short means positive". One-line negative reviews such as "Terrible hotel, dirty room and rude staff." are labelled positive.

**Idea.** Give the *same* GRU short training examples as well. For each **training** review we cut two random chunks of one or two consecutive sentences and give each chunk the label of the review it came from. Validation and test data are untouched (full reviews), and no chunk is ever taken from them.

**Caveat.** A chunk inherits the whole review's label, so some labels are noisy: a sentence from a positive review can be neutral or even negative.

In [7]:
# ---- Optional extension (GRU only): add short training examples ----------------------------------
def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if len(s.split()) >= 3]


def make_short_chunks(train_df, per_review=2, max_sentences=2, seed=SEED):
    # random chunks of 1-2 consecutive sentences, labelled like the review they were cut from
    rng = np.random.default_rng(seed)
    chunks, labels = [], []
    for text, label in zip(train_df["text"], train_df["label"]):
        sents = split_sentences(text)
        for _ in range(per_review):
            k = int(rng.integers(1, max_sentences + 1))
            if len(sents) <= k:
                continue
            start = int(rng.integers(0, len(sents) - k + 1))
            chunks.append(" ".join(sents[start:start + k]))
            labels.append(int(label))
    return chunks, np.array(labels, dtype="int32")


chunks, chunk_labels = make_short_chunks(data.train_df)          # TRAINING reviews only
X_chunks = pad([encode(clean_text(c).split(), data.vocab) for c in chunks])
X_train_aug = np.concatenate([data.X_train, X_chunks])
y_train_aug = np.concatenate([data.y_train, chunk_labels])
class_weight_aug = compute_class_weights(y_train_aug)

chunk_len = np.array([len(clean_text(c).split()) for c in chunks])
print(f"Added {len(chunks):,} short examples to the {len(data.y_train):,} training reviews "
      f"(median {int(np.median(chunk_len))} words, {100 * (chunk_len < 25).mean():.0f}% under 25 words).")
print("Examples:")
for c, y in list(zip(chunks, chunk_labels))[:3]:
    print(f"  [{'positive' if y else 'negative'}] {c[:110]}")
print("Class weights:", {k: round(v, 4) for k, v in class_weight_aug.items()})

Added 2,528 short examples to the 1,276 training reviews (median 22 words, 58% under 25 words).
Examples:
  [positive] Most major restaurants, Shopping, Sightseeing attractions within walking distance.
  [positive] Had a great view of river, lake, Wrigley Bldg. & Tribune Bldg.
  [positive] What I saw of Chicago was very pleasing, it has something for everyone.
Class weights: {0: 1.0011, 1: 0.9989}


In [8]:
keras.utils.set_random_seed(SEED)
model_short = build_gru()                     # the same architecture and settings as the main GRU
model_short.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                    loss="binary_crossentropy", metrics=["accuracy"])
early_stop = keras.callbacks.EarlyStopping(monitor=MONITOR, patience=PATIENCE, restore_best_weights=True)

start = time.perf_counter()
hist_short = model_short.fit(X_train_aug, y_train_aug,
                             validation_data=(data.X_val, data.y_val),      # validation = full reviews, as before
                             epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, class_weight=class_weight_aug,
                             callbacks=[early_stop], verbose=2).history
time_short = time.perf_counter() - start

epochs_short = len(hist_short["loss"])
best_short = int(np.argmin(hist_short["val_loss"])) + 1
print(f"\nEpochs completed: {epochs_short} | best epoch: {best_short} | training time {time_short:.1f} s")

Epoch 1/30


119/119 - 4s - 38ms/step - accuracy: 0.6285 - loss: 0.6464 - val_accuracy: 0.7563 - val_loss: 0.5192


Epoch 2/30


119/119 - 4s - 34ms/step - accuracy: 0.7973 - loss: 0.4670 - val_accuracy: 0.8188 - val_loss: 0.4159


Epoch 3/30


...

Epoch 12/30


119/119 - 5s - 39ms/step - accuracy: 0.9871 - loss: 0.0406 - val_accuracy: 0.9250 - val_loss: 0.4501


Epoch 13/30


119/119 - 5s - 39ms/step - accuracy: 0.9887 - loss: 0.0301 - val_accuracy: 0.9187 - val_loss: 0.4156



Epochs completed: 13 | best epoch: 8 | training time 56.4 s


**Reading the comparison.** The extension is judged on two things: it must not get worse on full reviews (validation accuracy), and it should handle one-line reviews (twelve probe sentences written before the extension was tried, and twenty new ones written after it was chosen and never used to pick anything). Test scores are shown once per model for completeness. With only 160 test reviews, differences of a few points are within noise, so they must not be used to pick a winner.

In [9]:
from IPython.display import display

# Twelve one-line reviews (6 negative, 6 positive), written before the extension was tried.
short_probes = [
    ("Terrible hotel, dirty room and rude staff.", 0), ("Worst hotel ever. Never again.", 0),
    ("Not good at all.", 0), ("I would not recommend this hotel.", 0),
    ("very bad place to visit never ever again visit to this place", 0),
    ("The room was not clean and the staff were not helpful.", 0),
    ("Great hotel, lovely staff and a spotless room.", 1), ("Loved it, we will definitely come back.", 1),
    ("Wonderful stay, very friendly staff.", 1), ("Amazing location and a beautiful room.", 1),
    ("Excellent service and a very comfortable bed.", 1), ("Highly recommend this hotel.", 1),
]
probe_X = pad([encode(clean_text(t).split(), data.vocab) for t, _ in short_probes])
probe_y = np.array([y for _, y in short_probes])

# Twenty more one-liners (10 negative, 10 positive), written after the extension was chosen.
# They were not used to pick anything, so they are a fairer test of one-line reviews.
fresh_probes = [
    ("Dirty rooms and rude staff, would not stay again.", 0), ("Horrible service and the room smelled awful.", 0),
    ("Very poor hotel, waste of money, never coming back.", 0), ("The bed was broken and nobody cared.", 0),
    ("Worst place I have stayed, bad room and bad food.", 0), ("bad hotel very dirty and noisy do not go there", 0),
    ("Overpriced and disappointing, the staff were unfriendly.", 0), ("We hated it. Cold room, no hot water, slow service.", 0),
    ("Not clean, not quiet, not worth the money.", 0), ("Terrible location and even worse customer service.", 0),
    ("Lovely hotel, friendly staff and a very clean room.", 1), ("Fantastic stay, we loved the view and the breakfast.", 1),
    ("Great value, comfortable bed and helpful staff.", 1), ("Perfect location and a beautiful, quiet room.", 1),
    ("Best hotel we have stayed in, will come again.", 1), ("good hotel very clean and nice staff, i like it", 1),
    ("Excellent service, delicious food and a great pool.", 1), ("We had a wonderful time, everything was perfect.", 1),
    ("Spotless room, comfy beds, friendly reception.", 1), ("Really enjoyed our stay, highly recommended.", 1),
]
fresh_X = pad([encode(clean_text(t).split(), data.vocab) for t, _ in fresh_probes])
fresh_y = np.array([y for _, y in fresh_probes])


def compare_row(name, m, epochs, seconds):
    val_prob = m.predict(data.X_val, verbose=0).ravel()
    probe_pred = (m.predict(probe_X, verbose=0).ravel() >= THRESHOLD).astype(int)
    fresh_pred = (m.predict(fresh_X, verbose=0).ravel() >= THRESHOLD).astype(int)
    test_m = compute_metrics(data.y_test, m.predict(data.X_test, verbose=0).ravel())   # TEST: evaluated once per model
    return {"model": name, "epochs": epochs, "train time (s)": round(seconds, 1),
            "val accuracy": round(float(((val_prob >= THRESHOLD) == data.y_val).mean()), 4),
            "one-liners correct (of 12)": int((probe_pred == probe_y).sum()),
            "negative one-liners (of 6)": int((probe_pred[probe_y == 0] == 0).sum()),
            "new one-liners correct (of 20)": int((fresh_pred == fresh_y).sum()),
            "new negative one-liners (of 10)": int((fresh_pred[fresh_y == 0] == 0).sum()),
            "test accuracy": round(test_m["accuracy"], 4), "test F1": round(test_m["f1"], 4),
            "test ROC-AUC": round(test_m["roc_auc"], 4)}, test_m


row_main, _ = compare_row("GRU (main)", model, epochs_run, train_time)
row_short, test_short = compare_row("GRU + short chunks (extension)", model_short, epochs_short, time_short)
display(pd.DataFrame([row_main, row_short]).set_index("model").T)

model_short.save(RESULTS / "gru_short_model.keras")
save_result(RESULTS / "gru_short_extension_metrics.json", "GRU + short chunks (extension)", test_short,
            training_time_sec=round(time_short, 2), trainable_params=n_params(model_short.trainable_weights),
            total_params=model_short.count_params(), epochs_run=epochs_short, best_epoch=best_short,
            extra={"validation_accuracy": row_short["val accuracy"],
                   "one_liners_correct_of_12": row_short["one-liners correct (of 12)"],
                   "new_one_liners_correct_of_20": row_short["new one-liners correct (of 20)"],
                   "note": "Optional GRU-only extension. Not part of the four-model comparison."})

model,GRU (main),GRU + short chunks (extension)
epochs,10.0000,13.0000
train time (s),13.7000,56.4000
val accuracy,0.9375,0.9437
one-liners correct (of 12),6.0000,11.0000
negative one-liners (of 6),0.0000,5.0000
new one-liners correct (of 20),10.0000,19.0000
new negative one-liners (of 10),0.0000,10.0000
test accuracy,0.8812,0.8875
test F1,0.8758,0.8889
test ROC-AUC,0.9367,0.9436


{'model': 'GRU + short chunks (extension)',
 'accuracy': 0.8875,
 'precision': 0.8780487804878049,
 'recall': 0.9,
 'f1': 0.8888888888888888,
 'roc_auc': 0.94359375,
 'training_time_sec': 56.44,
 'trainable_params': 347073,
 'total_params': 347073,
 'epochs_run': 13,
 'best_epoch': 8,
 'seed': 42,
 'max_len': 300,
 'vocab_size': 5000,
 'embed_dim': 64,
 'batch_size': 32,
 'learning_rate': 0.001,
 'threshold': 0.5,
 'confusion_matrix': {'tn': 70, 'fp': 10, 'fn': 8, 'tp': 72},
 'validation_accuracy': 0.9437,
 'one_liners_correct_of_12': 11,
 'new_one_liners_correct_of_20': 19,
 'note': 'Optional GRU-only extension. Not part of the four-model comparison.'}

## Try the model on your own reviews

Type or paste any hotel review and the trained models will say whether it looks **positive** or **negative**. The table shows the main GRU and the notebook 14 extension side by side.
Your text goes through exactly the same steps as the training data:

1. **Cleaning**: `clean_text()` lowercases, removes HTML, URLs, digits and punctuation, and keeps negations such as "not".
2. **Tokenisation**: the cleaned text is split into words.
3. **Encoding**: each word becomes its id in the training vocabulary (words the model never saw become `<OOV>`).
4. **Padding**: the ids are padded or cut to `MAX_LEN` tokens, so every review has the same length.

Each model returns `P(positive)`, a probability between 0 and 1. A review is called *Positive* when it is at least the threshold (0.5).

Keep in mind that the models were trained on only 1,276 Chicago hotel reviews, each at least 25 words long (median about 130). Their probabilities are over-confident (see the error analysis), and mixed reviews, which praise one thing and criticise another, are the hardest cases.

**Very short texts.** The main GRU is unreliable for a review shorter than any training review: in our checks one-line reviews such as "Terrible hotel, dirty room and rude staff." were labelled *positive*. The function prints a note when this happens. The notebook 14 extension was trained on short examples, so compare the two columns.

In [10]:
from IPython.display import display


def predict_sentiment(model, texts, threshold=0.5):
    # `model` is one Keras model, or a dict {"name": model} to get one column per model side by side
    models = model if isinstance(model, dict) else {None: model}
    if isinstance(texts, str):
        texts = [texts]
    texts = list(texts)
    cleaned = [clean_text(t) for t in texts]
    # same steps as training: clean -> split into words -> encode with the training vocabulary -> pad
    X = pad([encode(c.split(), data.vocab) for c in cleaned])

    min_len = int(data.train_df["tokens"].map(len).min())      # the shortest original training review
    n_short = sum(len(c.split()) < min_len for c in cleaned)
    if n_short:
        print(f"Note: {n_short} review(s) have fewer than {min_len} words, shorter than any original training review. "
              "The main GRU is unreliable for very short text.")

    no_words = np.array([c == "" for c in cleaned])           # nothing left after cleaning (e.g. "???")
    probs = {name: m.predict(X, verbose=0).ravel() for name, m in models.items()}
    suffix = lambda name: "" if name is None else f" {name}"
    table = {"review": texts}
    for name, prob in probs.items():
        table["P(positive)" + suffix(name)] = np.where(no_words, np.nan, np.round(prob, 3))
    for name, prob in probs.items():
        label = np.where(prob >= threshold, "Positive", "Negative")
        table["prediction" + suffix(name)] = np.where(no_words, "No usable text", label)
    return pd.DataFrame(table)


models = {"GRU": model, "GRU + short": model_short}           # main model and the notebook 14 extension

my_reviews = [
    # clearly positive
    "Absolutely wonderful stay! The staff were friendly and helpful, the room was spotless and very comfortable, and the view of the lake from our window was amazing. Breakfast was delicious and the location is perfect for exploring the city. I would happily come back and recommend it to friends.",
    # clearly negative
    "Terrible experience. The room was dirty, the carpet was stained and the bathroom smelled of mold. The staff were rude and did nothing when we complained, and the noise from the street kept us awake all night. It was the worst hotel I have ever stayed in and a waste of money.",
    # negation
    "Not good at all. I would not recommend this hotel to anyone. The room was not clean, the staff were not helpful and the breakfast was not worth the price. We did not get any apology when we complained, and we will not be coming back.",
    # mixed
    "Great location but the bed was uncomfortable and very noisy. We could walk to the shops and restaurants in a few minutes, which was perfect. Unfortunately the mattress was hard and the street noise made it hard to sleep, so we were tired the whole trip.",
    # mixed
    "The breakfast was excellent and the staff were lovely, however the room was tiny and smelled of smoke. The lobby and the restaurant looked great, but our room was old and the shower barely worked. Nice people, disappointing room.",
]

pd.set_option("display.max_colwidth", 90)
display(predict_sentiment(models, my_reviews))

# Very short reviews: the main GRU is unreliable here; the extension model was trained on short examples.
short_reviews = [
    "very bad place to visit never ever again visit to this place",
    "Not good at all.",
    "Worst hotel ever. Never again.",
    "Great hotel, lovely staff and a spotless room.",
]
display(predict_sentiment(models, short_reviews))

,review,P(positive) GRU,P(positive) GRU + short,prediction GRU,prediction GRU + short
0,"Absolutely wonderful stay! The staff were friendly and helpful, the room was spotless ...",1.000,1.0,Positive,Positive
1,"Terrible experience. The room was dirty, the carpet was stained and the bathroom smell...",0.001,0.0,Negative,Negative
2,"Not good at all. I would not recommend this hotel to anyone. The room was not clean, t...",0.013,0.0,Negative,Negative
3,Great location but the bed was uncomfortable and very noisy. We could walk to the shop...,0.488,0.0,Negative,Negative
4,"The breakfast was excellent and the staff were lovely, however the room was tiny and s...",0.225,0.0,Negative,Negative


Note: 4 review(s) have fewer than 25 words, shorter than any original training review. The main GRU is unreliable for very short text.


,review,P(positive) GRU,P(positive) GRU + short,prediction GRU,prediction GRU + short
0,very bad place to visit never ever again visit to this place,0.997,0.994,Positive,Positive
1,Not good at all.,1.000,0.042,Positive,Negative
2,Worst hotel ever. Never again.,0.998,0.000,Positive,Negative
3,"Great hotel, lovely staff and a spotless room.",1.000,0.999,Positive,Positive


In [11]:
# Interactive: type one review (leave empty to skip)
try:
    text = input("Enter a hotel review: ").strip()
except Exception:            # a non-interactive run (for example nbconvert) cannot ask for input, so skip
    text = ""
if text:
    display(predict_sentiment(models, [text]))

---
*Notebook 14 of 14 · Previous: 13 Handover · Next: none (the end)*